# Community Detection Example (Mall Customers Dataset)

Here it is demonstrated how to use the `LabelPropagation` module from the CMOR-438 library to detect communities in a customer similarity graph.
In this example, the Mall Customers dataset is converted into a graph by connecting similar shoppers, then communities are discovered by label propagation.

**Goal: Find communities of similar customers by treating the dataset as a similarity network.**

## 1. Setup and Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys, os

# Add the algorithm folder to path
NOTEBOOK_DIR = os.path.abspath('')
sys.path.insert(0, NOTEBOOK_DIR)

# Data lives two levels up from the algorithm folder
DATA_DIR = os.path.join(NOTEBOOK_DIR, '..', '..', 'data')
# Also add pca and k_means folders for the visualisation cells
sys.path.insert(0, os.path.join(NOTEBOOK_DIR, '..', 'pca'))
sys.path.insert(0, os.path.join(NOTEBOOK_DIR, '..', 'k_means_clustering'))

from community_detection import LabelPropagation
from pca import PCA
from k_means_clustering import KMeans
from sklearn.preprocessing import StandardScaler

mall = pd.read_csv(os.path.join(DATA_DIR, 'Mall_Customers.csv'))
MALL_FEATURES = ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']
X_raw = mall[MALL_FEATURES].values.astype(float)
X = StandardScaler().fit_transform(X_raw)

print(f"Dataset loaded: {mall.shape[0]} samples, {len(MALL_FEATURES)} features.")

## 2. Build the Similarity Graph

Connect customers whose standardised feature distance is below the threshold.

In [ ]:
threshold = 0.8
dists = np.sqrt(((X[:,None,:] - X[None,:,:])**2).sum(axis=2))
A = (dists < threshold).astype(float)
np.fill_diagonal(A, 0)

print(f'Graph edges:    {int(A.sum()//2)}')
print(f'Average degree: {A.sum(axis=1).mean():.1f}')
print(f'Density:        {A.sum() / (len(X)*(len(X)-1)):.3f}')

## 3. Run Label Propagation

In [ ]:
lp = LabelPropagation(max_iter=100, random_state=42).fit(A)
print(f'Communities found: {lp.n_communities_}')
print(f'Converged in:      {lp.n_iter_} iterations')
print(f'Modularity:        {lp.modularity(A):.4f}  (>0.3 indicates meaningful structure)')

sizes = {c: len(v) for c, v in lp.get_communities().items()}
print(f'Community sizes:   {dict(sorted(sizes.items(), key=lambda x: -x[1]))}')

## 4. Results and Visualisation

Communities in PCA space and in Income vs Spending space.

In [ ]:
pca = PCA(n_components=2).fit(X)
X_pca = pca.transform(X)

lp_labels = lp.labels_
n_comm = lp.n_communities_
cmap_lp = plt.cm.get_cmap('tab20', n_comm)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for c in range(n_comm):
    mask = lp_labels==c
    if mask.sum() > 0:
        axes[0].scatter(X_pca[mask,0], X_pca[mask,1], color=cmap_lp(c), s=30, alpha=0.75, label=f'C{c} (n={mask.sum()})')
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')
axes[0].set_title(f'Label Propagation - {n_comm} Communities (PCA view)', fontweight='bold')
axes[0].legend(fontsize=7, ncol=2)

comm_df = pd.DataFrame({'Income': X_raw[:,1], 'Spending': X_raw[:,2], 'Community': lp_labels})
for c in range(n_comm):
    sub = comm_df[comm_df['Community']==c]
    axes[1].scatter(sub['Income'], sub['Spending'], color=cmap_lp(c), s=30, alpha=0.75)
    cx, cy = sub['Income'].mean(), sub['Spending'].mean()
    axes[1].annotate(f'C{c}', (cx,cy), fontsize=8, fontweight='bold', ha='center', va='center',
                     bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7))
axes[1].set_xlabel('Annual Income (k$)'); axes[1].set_ylabel('Spending Score')
axes[1].set_title('Label Propagation - Income vs Spending', fontweight='bold')
plt.tight_layout(); plt.show()